In [ ]:
%pip install mlflow dvc fastapi "uvicorn[standard]" pytest httpx

In [ ]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from mlflow import MlflowClient
from mlflow.models import infer_signature

pd.set_option("display.max_columns", None)

In [ ]:
def find_project_root(start_path: Path) -> Path:

    start_path = start_path.resolve()

    for candidate in [
        start_path,
        *start_path.parents
    ]:

        expected_model = (
            candidate
            / "models"
            / "final_tuned_churn_model.joblib"
        )

        if expected_model.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root."
    )


project_root = find_project_root(Path.cwd())

print("Project root:")
print(project_root)

In [ ]:
final_model_path = (
    project_root
    / "models"
    / "final_tuned_churn_model.joblib"
)


metadata_path = (
    project_root
    / "models"
    / "final_tuned_churn_model_metadata.json"
)


model_data_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_model.csv"
)


business_threshold_path = (
    project_root
    / "models"
    / "business_threshold_policy.json"
)


threshold_comparison_path = (
    project_root
    / "reports"
    / "model_results"
    / "business_threshold_comparison.csv"
)


explainability_metadata_path = (
    project_root
    / "reports"
    / "explainability"
    / "explainability_metadata.json"
)


rag_metadata_path = (
    project_root
    / "reports"
    / "rag"
    / "rag_metadata.json"
)


for path in [
    final_model_path,
    metadata_path,
    model_data_path
]:
    print(
        path.name,
        "→",
        path.exists()
    )

In [ ]:
final_model = joblib.load(
    final_model_path
)


with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as file:

    model_metadata = json.load(file)


model_df = pd.read_csv(
    model_data_path
)


X = model_df.drop(
    columns=["churn_value"]
)

y = model_df["churn_value"]


print("Model:")
print(type(final_model))

print("\nFeatures:")
print(X.shape)

print("\nMetadata:")
display(model_metadata)

In [ ]:
input_example = (
    X.head(5).copy()
)


example_probabilities = (
    final_model
    .predict_proba(
        input_example
    )[:, 1]
)


print(
    "Example probabilities:"
)

print(
    example_probabilities
)

In [ ]:
mlops_directory = (
    project_root
    / "mlops"
)


mlflow_artifact_directory = (
    mlops_directory
    / "mlflow_artifacts"
)


mlops_directory.mkdir(
    parents=True,
    exist_ok=True
)


mlflow_artifact_directory.mkdir(
    parents=True,
    exist_ok=True
)


mlflow_database_path = (
    mlops_directory
    / "mlflow.db"
)


print(
    "MLflow database:"
)

print(
    mlflow_database_path
)

In [ ]:
tracking_uri = (
    "sqlite:///"
    +
    mlflow_database_path
    .resolve()
    .as_posix()
)


mlflow.set_tracking_uri(
    tracking_uri
)


EXPERIMENT_NAME = (
    "telco_customer_churn"
)


REGISTERED_MODEL_NAME = (
    "telco_churn_calibrated_xgboost"
)


experiment = (
    mlflow.get_experiment_by_name(
        EXPERIMENT_NAME
    )
)


if experiment is None:

    experiment_id = (
        mlflow.create_experiment(
            name=EXPERIMENT_NAME,

            artifact_location=(
                mlflow_artifact_directory
                .resolve()
                .as_uri()
            )
        )
    )

else:

    experiment_id = (
        experiment.experiment_id
    )


mlflow.set_experiment(
    EXPERIMENT_NAME
)


print(
    "Tracking URI:",
    mlflow.get_tracking_uri()
)

print(
    "Experiment:",
    EXPERIMENT_NAME
)

print(
    "Experiment ID:",
    experiment_id
)

In [ ]:
def extract_numeric_metrics(
    dictionary,
    prefix=""
):

    metrics = {}

    for key, value in dictionary.items():

        metric_name = (
            f"{prefix}_{key}"
            if prefix
            else key
        )


        if isinstance(
            value,
            (int, float)
        ) and not isinstance(
            value,
            bool
        ):

            if np.isfinite(value):

                metrics[
                    metric_name
                ] = float(value)


        elif isinstance(
            value,
            dict
        ):

            metrics.update(
                extract_numeric_metrics(
                    value,
                    prefix=metric_name
                )
            )


    return metrics

numeric_metadata = (
    extract_numeric_metrics(
        model_metadata
    )
)


display(
    pd.DataFrame(
        numeric_metadata.items(),
        columns=[
            "metric",
            "value"
        ]
    )
)

In [ ]:
run_parameters = {

    "model_family":
        model_metadata.get(
            "model_name",
            "XGBoost"
        ),

    "probability_source":
        model_metadata.get(
            "probability_source",
            "Sigmoid calibrated"
        ),

    "selected_threshold":
        str(
            model_metadata.get(
                "selected_threshold",
                "Unknown"
            )
        ),

    "calibration":
        "sigmoid",

    "task":
        "binary_classification",

    "target":
        "churn_value",

    "number_of_features":
        X.shape[1],

    "training_pipeline":
        (
            "preprocessing + "
            "XGBoost + "
            "probability calibration"
        )
}


display(
    pd.DataFrame(
        run_parameters.items(),
        columns=[
            "parameter",
            "value"
        ]
    )
)

In [ ]:
signature = infer_signature(
    input_example,
    example_probabilities
)


print(signature)

In [ ]:
trusted_types = [
    "numpy.dtype",
    "numpy.number",
    "sklearn.calibration._CalibratedClassifier",
    "sklearn.calibration._SigmoidCalibration",
    "sklearn.compose._column_transformer.make_column_selector",
    "sklearn.model_selection._split.StratifiedKFold",
    "xgboost.core.Booster",
    "xgboost.sklearn.XGBClassifier",
]

with mlflow.start_run(
    run_name=(
        "final_calibrated_xgboost"
    )
) as run:

    run_id = run.info.run_id

    # -------------------------
    # Parameters
    # -------------------------
    mlflow.log_params(
        run_parameters
    )

    # -------------------------
    # Metrics
    # -------------------------
    for metric_name, value in (
        numeric_metadata.items()
    ):

        try:
            mlflow.log_metric(
                metric_name,
                value
            )
        except Exception:
            pass

    # -------------------------
    # Tags
    # -------------------------
    mlflow.set_tags(
        {
            "project":
                "Customer Churn",

            "model_status":
                "final",

            "model_type":
                "calibrated_xgboost",

            "calibration_method":
                "sigmoid",

            "purpose":
                "retention_risk_prediction"
        }
    )

    # -------------------------
    # Existing artifacts
    # -------------------------
    artifact_files = [
        metadata_path,
        business_threshold_path,
        threshold_comparison_path,
        explainability_metadata_path,
        rag_metadata_path
    ]

    for artifact_path in (
        artifact_files
    ):

        if artifact_path.exists():
            mlflow.log_artifact(
                str(
                    artifact_path
                ),

                artifact_path=(
                    "project_metadata"
                )
            )

    # -------------------------
    # Log + register model
    # -------------------------
    logged_model = (
        mlflow.sklearn.log_model(

            sk_model=final_model,

            name="model",

            signature=signature,

            input_example=(
                input_example
            ),

            registered_model_name=(
                REGISTERED_MODEL_NAME
            ),

            skops_trusted_types=trusted_types
        )
    )

print(
    "MLflow Run ID:"
)

print(run_id)


In [ ]:
client = MlflowClient()


model_versions = (
    client.search_model_versions(
        f"name='{REGISTERED_MODEL_NAME}'"
    )
)


for version in model_versions:

    print(
        "Version:",
        version.version,
        "| Run:",
        version.run_id
    )

In [ ]:
current_version = None


for version in model_versions:

    if version.run_id == run_id:

        current_version = (
            version.version
        )

        break


if current_version is None:

    raise RuntimeError(
        "Could not find registered model version."
    )


client.set_registered_model_alias(

    REGISTERED_MODEL_NAME,

    "champion",

    current_version
)


print(
    "Champion version:",
    current_version
)

In [ ]:
champion_model_uri = (
    f"models:/"
    f"{REGISTERED_MODEL_NAME}"
    f"@champion"
)


champion_model = (
    mlflow.sklearn.load_model(
        champion_model_uri
    )
)


champion_probabilities = (
    champion_model
    .predict_proba(
        input_example
    )[:, 1]
)


print(
    "Registry probabilities:"
)

print(
    champion_probabilities
)

In [ ]:
maximum_difference = (
    np.max(
        np.abs(
            example_probabilities
            -
            champion_probabilities
        )
    )
)


print(
    "Maximum difference:",
    maximum_difference
)


assert np.allclose(
    example_probabilities,
    champion_probabilities,
    atol=1e-8
)


print(
    "Registered model validation passed."
)

In [ ]:
print("=" * 65)

print(
    "MLFLOW SETUP COMPLETE"
)

print("=" * 65)

print(
    "Experiment:",
    EXPERIMENT_NAME
)

print(
    "Run ID:",
    run_id
)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME
)

print(
    "Version:",
    current_version
)

print(
    "Deployment alias:",
    "champion"
)

print(
    "Model URI:",
    champion_model_uri
)